# dWGS — Part 8: Data Requirements (NEY GMS → NW GMS, RGL to SGL)

NHS England is transitioning centralised Whole Genome Sequencing (WGS) to a distributed
model (dWGS): each NHS GMS geography's **Requesting Genomic Laboratory (RGL)** submits
DNA samples directly to a **Sequencing Genomic Laboratory (SGL)**. This notebook covers
a specific subcontracted arrangement: **North East and Yorkshire GMS acting as RGL**,
submitting samples and a digital manifest to **NW GMS acting as SGL**, whose inbound
order pathway is the same NW Regional Integration Engine (RIE) → iGene ORM^O01
interface `03`-`07` already work with.

Three source documents define the requirement (all in `NotGit/`, gitignored per this
repo's convention, referenced here by name only):

- **`RGL to SGL SOP v0.4.docx`** — NHS England's SOP; Appendix 3 defines the 37-column
  national Digital Manifest CSV.
- **`dWGS Manifest CSV to HL7v2 ORM Transformation Specification v0.1.docx`** — NW GMS's
  draft spec turning that manifest into an ORM^O01 message, plus a 5-column local
  extension NEY GMS adds for NW GMS's benefit.
- **`dWGS Sample Manifest HL7v2 Data Mapping v0.1.xlsx`** — the field-by-field working
  sheet those tables were drawn from.

This is a preparation notebook: the sample data (`Input/dWGS.csv`) and the three-way
field mapping below.

## Sample manifest data

In [1]:
import pandas as pd

dwgs_df = pd.read_csv("Input/dWGS.csv", dtype=str, keep_default_na=False)
print(f"{len(dwgs_df)} rows, {len(dwgs_df.columns)} columns")
dwgs_df

6 rows, 42 columns


,referral_id,clinical_indication_test_type_id,patient_nhs_number,patient_ngis_id,patient_date_of_birth,ordering_entity_id,glh_laboratory_id,primary_sample_received_date,primary_sample_id_as_received_by_glh,primary_sample_id_in_glh_lims,...,gmc_rack_well,dna_extraction_protocol,prolonged_sample_storage,retrospective_sample,approved_by,patient_forename,patient_surname,family_structure,participant_type,clinical_information
0,r2026000201,R14.1,9737383222,p2026000101,1978-01-17,RR8,RR8,2026-08-20,S26-2000,S26-2000,...,A01,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Rob,Leeds,Singleton,Proband,Suspected inherited neurodevelopmental condition.
1,r2026000202,R59.1,9737873971,p2026000102,1947-04-27,RTR,RTR,2026-08-20,S26-2001,S26-2001,...,A02,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Allanys,Middlesborough,Duo,Proband,
2,r2026000202,R59.1,9999999603,p2026000103,1984-11-06,RTR,RTR,2026-08-20,S26-2002,S26-2002,...,A03,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Fourteen,Editestpatient,Duo,Family Member,
3,r2026000203,R27.3,9737383362,p2026000104,1989-07-01,RNN,RNN,2026-08-20,S26-2003,S26-2003,...,A04,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Gilly,Brough,Trio,Proband,Patient has intellectual disability
4,r2026000203,R27.3,9999999581,p2026000105,1960-01-01,RNN,RNN,2026-08-20,S26-2004,S26-2004,...,A05,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Thirteen,Editestpatient,Trio,Family Member,
5,r2026000203,R27.3,9999999514,p2026000106,1988-01-14,RNN,RNN,2026-08-20,S26-2005,S26-2005,...,A06,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Six,Editestpatient,Trio,Family Member,


## Field mapping: CSV → HL7v2 → FHIR

All 42 manifest fields (37 national, from `RGL to SGL SOP v0.4` Appendix 3, plus 5 NEY
local-extension fields from the transformation spec Section 7.2). Fields the
transformation spec marks `HL7v2 Mapping Required = FALSE` are left blank in both the
`hl7v2_field` and `fhir_field` columns — they're carried in the CSV but not transmitted
onward.

The `fhir_field` column is this notebook's own addition — neither source document
mentions FHIR. Identifiers/profiles reused from `03`-`07` are shown as-is;
`family_structure` and `participant_type` have no home in the current NW-GMSA IG and are
marked `(proposed)`, and the NGIS identifier system is marked `(TBC)` since no published
FHIR system for it was found in either source document.

In [2]:
FIELD_MAPPING = [
    # csv_field, hl7v2_field, fhir_field - in the same column order as Input/dWGS.csv
    ("referral_id", "OBX-5 (OBX-3=NGIS_REFERRAL_ID)", "ServiceRequest.identifier (TBC system)"),
    ("clinical_indication_test_type_id", "OBR-4.1", "ServiceRequest.code.coding (England-GenomicTestDirectory)"),
    ("patient_nhs_number", "PID-3 (NH)", "Patient.identifier (NHS number)"),
    ("patient_ngis_id", "PID-3 (NGIS)", "Patient.identifier (TBC system)"),
    ("patient_date_of_birth", "PID-7.1", "Patient.birthDate"),
    ("ordering_entity_id", "", ""),
    ("glh_laboratory_id", "", ""),
    ("primary_sample_received_date", "", ""),
    ("primary_sample_id_as_received_by_glh", "", ""),
    ("primary_sample_id_in_glh_lims", "", ""),
    ("primary_sample_type", "", ""),
    ("primary_sample_state", "", ""),
    ("received_sample_topography", "", ""),
    ("received_sample_morphology", "", ""),
    ("received_sample_tumour_content_%", "", ""),
    ("received_sample_comments", "", ""),
    ("received_sample_collection_date", "", ""),
    ("dispatched_sample_id_in_glh_lims", "", ""),
    ("dispatched_sample_lsid", "SPM-2.1 and OBX-5 (OBX-3=DISPATCHED_SAMPLE_LSID)", "Specimen.container.identifier"),
    ("dispatched_sample_type", "", ""),
    ("dispatched_sample_state", "SPM-4.1", "Specimen.type (text-only - DNA)"),
    ("dispatched_sample_volume_(ul)", "", ""),
    ("laboratory_remaining_volume_banked_(ul)", "", ""),
    ("glh_concentration_(ng/ul)", "", ""),
    ("glh_od260/280", "", ""),
    ("glh_din_value", "", ""),
    ("glh_percentage_DNA_over_23kb", "", ""),
    ("glh_qc_status", "", ""),
    ("glh_sample_dispatch_date", "", ""),
    ("glh_sample_consignment_number", "", ""),
    ("plating_organisation", "", ""),
    ("gmc_rack_id", "", ""),
    ("gmc_rack_well", "", ""),
    ("dna_extraction_protocol", "", ""),
    ("prolonged_sample_storage", "", ""),
    ("retrospective_sample", "", ""),
    ("approved_by", "", ""),
    ("patient_forename", "PID-5.2", "Patient.name.given"),
    ("patient_surname", "PID-5.1", "Patient.name.family"),
    ("family_structure", "OBX-5 (OBX-3=FAMILY_STRUCTURE)", "ServiceRequest.extension (proposed)"),
    ("participant_type", "OBX-5 (OBX-3=PARTICIPANT_TYPE)", "ServiceRequest.extension (proposed)"),
    ("clinical_information", "NTE-3", "ServiceRequest.note"),
]

assert [f[0] for f in FIELD_MAPPING] == list(dwgs_df.columns), \
    "field_mapping order must match Input/dWGS.csv's own column order"

field_mapping_df = pd.DataFrame(FIELD_MAPPING, columns=["csv_field", "hl7v2_field", "fhir_field"])
field_mapping_df

,csv_field,hl7v2_field,fhir_field
0,referral_id,OBX-5 (OBX-3=NGIS_REFERRAL_ID),ServiceRequest.identifier (TBC system)
1,clinical_indication_test_type_id,OBR-4.1,ServiceRequest.code.coding (England-GenomicTes...
2,patient_nhs_number,PID-3 (NH),Patient.identifier (NHS number)
3,patient_ngis_id,PID-3 (NGIS),Patient.identifier (TBC system)
4,patient_date_of_birth,PID-7.1,Patient.birthDate
5,ordering_entity_id,,
6,glh_laboratory_id,,
7,primary_sample_received_date,,
8,primary_sample_id_as_received_by_glh,,
9,primary_sample_id_in_glh_lims,,
